# Confronto: diffusore from-scratch (04b) vs diffusore fine-tuned (03b)

Notebook di sola analisi: legge esclusivamente gli artefatti gia' prodotti da

- `03b_Finetuning_StableDiffusion2.1_filtered.ipynb` -> Stable Diffusion 2.1 **fine-tuned** sulle mammografie RSNA (100 inference step, filtro adattivo, 1361 immagini/classe);
- `04b_LDM_extra1361.ipynb` -> Latent Diffusion Model Keras addestrato **from scratch** (VAE + UNet, 80000 step, filtro adattivo, 1361 immagini per la sola classe positiva).

Nessuna immagine viene generata qui e non serve la GPU. I due esperimenti non sono perfettamente simmetrici: il modello from-scratch ha prodotto finora soltanto la classe positiva (tumorale), mentre il fine-tuned ha prodotto entrambe le classi. Dove il confronto a tre vie (reali / from-scratch / fine-tuned) non e' possibile, il notebook lo segnala invece di nasconderlo.

Contenuto:

1. griglia reali vs from-scratch vs fine-tuned (classe positiva);
2. griglia reali vs fine-tuned (classe negativa, from-scratch non disponibile);
3. metriche finali sul test set (FID, Inception Score, PRDC);
4. effetto del filtro adattivo sul validation set (raw vs filtrate);
5. traiettoria di selezione del checkpoint durante il training;
6. costo energetico e CO2 stimati delle due pipeline;
7. tabella di riepilogo.


## 1. Percorsi progetto e sorgenti dei due esperimenti

In [ ]:
from pathlib import Path
import sys

PROJECT_NAME = "MammoDiffusion"
RESULTS_STAGE_NAME = "04c_Confronto_FromScratch_vs_FineTuned"

# Se serve su Colab/Drive:
# PROJECT_ROOT_OVERRIDE = Path("/content/drive/MyDrive/MammoDiffusion")
PROJECT_ROOT_OVERRIDE = None


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name:
            return candidate
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    for candidate in [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
    ]:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Non riesco a trovare la root MammoDiffusion. "
        "Esegui il notebook dalla repo o imposta PROJECT_ROOT_OVERRIDE."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
SYNTHETIC_DIR = DATA_DIR / "synthetic"

# Risultati gia' prodotti dai due notebook confrontati (sola lettura)
RESULTS_FT_DIR = PROJECT_ROOT / "results" / "03b_finetuning_filtered"
RESULTS_FS_DIR = PROJECT_ROOT / "results" / "04b_ldm_keras_v2_extra1361"

# Output di questo notebook
RESULTS_DIR = PROJECT_ROOT / "results" / RESULTS_STAGE_NAME
RESULTS_PLOTS_DIR = RESULTS_DIR / "plots"
RESULTS_METRICS_DIR = RESULTS_DIR / "metrics"
for directory in [RESULTS_DIR, RESULTS_PLOTS_DIR, RESULTS_METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("RESULTS_FT_DIR   :", RESULTS_FT_DIR, "(03b, fine-tuned)")
print("RESULTS_FS_DIR   :", RESULTS_FS_DIR, "(04b, from scratch)")
print("RESULTS_DIR      :", RESULTS_DIR)


## 2. Immagini reali e sintetiche disponibili

Le immagini reali di confronto sono quelle del **test set** (mai usate per scegliere checkpoint o calibrare i filtri in nessuno dei due notebook). Le sintetiche sono le versioni gia' **filtrate** (1361 immagini/classe) prodotte dai due notebook e salvate in `data/synthetic/`:

- `data/synthetic/fine_tuned/{positive,negative}` -> 03b;
- `data/synthetic/fromscratch/positive` -> 04b (la classe negativa non e' ancora stata generata).

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


def count_images(directory):
    directory = Path(directory)
    if not directory.is_dir():
        return 0
    return sum(1 for p in directory.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


REAL_DIRS = {
    "positive": DATA_PROCESSED_DIR / "test" / "1",
    "negative": DATA_PROCESSED_DIR / "test" / "0",
}

# Fine-tuned (03b): Stable Diffusion 2.1, entrambe le classi disponibili.
FINETUNED_DIRS = {
    "positive": SYNTHETIC_DIR / "fine_tuned" / "positive",
    "negative": SYNTHETIC_DIR / "fine_tuned" / "negative",
}

# From scratch (04b): LDM Keras, solo la classe positiva generata finora.
FROMSCRATCH_DIRS = {
    "positive": SYNTHETIC_DIR / "fromscratch" / "positive",
}

print("Reali (test):")
for label, directory in REAL_DIRS.items():
    print(f"  {label:9s}: {count_images(directory):4d}  ({directory})")

print("\nFine-tuned (03b, filtrate):")
for label, directory in FINETUNED_DIRS.items():
    print(f"  {label:9s}: {count_images(directory):4d}  ({directory})")

print("\nFrom scratch (04b, filtrate):")
for label, directory in FROMSCRATCH_DIRS.items():
    print(f"  {label:9s}: {count_images(directory):4d}  ({directory})")
print("  negative : non disponibile, il modello from-scratch ha generato solo la classe positiva")


## 3. Griglia principale: reali vs from-scratch vs fine-tuned (classe positiva)

Prime 5 immagini di ciascuna cartella, in ordine alfabetico di nome file (nessun campionamento casuale). E' la classe su cui entrambe le pipeline hanno generato immagini, quindi il confronto a tre vie e' possibile solo qui.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

N_SAMPLES = 3


def first_image_paths(directory, n_samples):
    return sorted(Path(directory).glob("*.png"))[:n_samples]


positive_groups = {
    "Reali (test)": first_image_paths(REAL_DIRS["positive"], N_SAMPLES),
    "From scratch (04b)": first_image_paths(FROMSCRATCH_DIRS["positive"], N_SAMPLES),
    "Fine-tuned (03b)": first_image_paths(FINETUNED_DIRS["positive"], N_SAMPLES),
}

fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(9, 3 * N_SAMPLES))
for col, (group_name, paths) in enumerate(positive_groups.items()):
    for row, path in enumerate(paths):
        with PILImage.open(path) as image:
            axes[row, col].imshow(np.asarray(image.convert("L")), cmap="gray", vmin=0, vmax=255)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
    axes[0, col].set_title(group_name, fontsize=12)

fig.suptitle("Classe positiva: reali vs from-scratch vs fine-tuned", fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.96))
output_path = RESULTS_PLOTS_DIR / "grid_real_vs_fromscratch_vs_finetuned_positive.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Salvato:", output_path)


## 5. Metriche finali sul test set

Legge `final_test_metrics.json` (03b) e `final_filtered_vs_test.json` (04b), limitandosi alla **classe positiva** (tumorale): e' l'unica generata dal modello from-scratch ed e' la classe obiettivo dell'augmentation, quindi e' l'unico confronto realmente alla pari. Stesso backend di valutazione (`generative_evaluator.py`) e stesso riferimento reale (test set) per entrambi, quindi FID/IS/PRDC sono direttamente comparabili.

In [ ]:
import json

import pandas as pd

with open(RESULTS_FT_DIR / "metrics" / "final_test_metrics.json", encoding="utf-8") as handle:
    ft_final = json.load(handle)

with open(RESULTS_FS_DIR / "metrics" / "final_filtered_vs_test.json", encoding="utf-8") as handle:
    fs_final = json.load(handle)

# Confronto limitato alla classe positiva (tumorale): e' l'unica generata dal
# modello from-scratch (04b) ed e' la classe obiettivo dell'augmentation.
final_rows = []

ft_metrics = ft_final["per_class"]["positive"]
final_rows.append({
    "pipeline": "Fine-tuned (03b)",
    "class": "positive",
    "FID": ft_metrics["FID"],
    "IS_mean": ft_metrics["IS_mean"],
    "IS_std": ft_metrics["IS_std"],
    "precision": ft_metrics["precision"],
    "recall": ft_metrics["recall"],
    "density": ft_metrics["density"],
    "coverage": ft_metrics["coverage"],
    "n_generated": ft_metrics["n_generated"],
    "n_real_reference": ft_metrics["n_real_reference"],
})

fs_metrics = fs_final["metrics"]
final_rows.append({
    "pipeline": "From scratch (04b)",
    "class": "positive",
    "FID": fs_metrics["FID"],
    "IS_mean": fs_metrics["IS_mean"],
    "IS_std": fs_metrics["IS_std"],
    "precision": fs_metrics["precision"],
    "recall": fs_metrics["recall"],
    "density": fs_metrics["density"],
    "coverage": fs_metrics["coverage"],
    "n_generated": fs_metrics["n_synthetic_filtered"],
    "n_real_reference": fs_metrics["n_real_test"],
})

df_final_metrics = pd.DataFrame(final_rows)
output_csv = RESULTS_METRICS_DIR / "final_test_metrics_comparison.csv"
df_final_metrics.to_csv(output_csv, index=False)
print("Salvato:", output_csv)
df_final_metrics

In [ ]:
metric_specs = [
    ("FID", "FID (piu' basso = meglio)"),
    ("IS_mean", "Inception Score"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("density", "Density"),
    ("coverage", "Coverage"),
]

df_final_metrics["serie"] = df_final_metrics["pipeline"].str.replace(
    r"\s*\(0[34]b\)", "", regex=True
)
colors = {
    "Fine-tuned": "#2e8b57",
    "From scratch": "#c0392b",
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (column, title) in zip(axes.ravel(), metric_specs):
    bars = ax.bar(
        df_final_metrics["serie"],
        df_final_metrics[column],
        color=[colors.get(serie, "#888888") for serie in df_final_metrics["serie"]],
    )
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=8)
    # Margine in alto cosi' le etichette delle barre non toccano il bordo del grafico
    top = max(df_final_metrics[column].max(), 0)
    ax.set_ylim(top=top * 1.15)
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Metriche finali sul test set (classe positiva): from-scratch (04b) vs fine-tuned (03b)", fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.95))
output_path = RESULTS_PLOTS_DIR / "final_test_metrics_comparison.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Salvato:", output_path)

## 6. Figura compatta per la slide D1

Questa figura affianca, non sostituisce, il plot 2x3 della cella precedente.

In [ ]:
ft_color = "#4CAF50"
fs_color = "#E53935"
navy = "#15335B"
grid_color = "#E2E8F0"
bar_navy = "#15335B"
label_color = "#1E293B"
tick_color = "#475569"

plt.rcParams["font.family"] = "DejaVu Sans"

ft_row = df_final_metrics[df_final_metrics["pipeline"] == "Fine-tuned (03b)"].iloc[0]
fs_row = df_final_metrics[df_final_metrics["pipeline"] == "From scratch (04b)"].iloc[0]

fig = plt.figure(figsize=(9.6, 7.4), dpi=160)
fig.patch.set_facecolor("white")
gs = fig.add_gridspec(2, 2, height_ratios=[2.2, 1], hspace=0.45, wspace=0.25)

ax_prdc = fig.add_subplot(gs[0, :])
ax_fid = fig.add_subplot(gs[1, 0])
ax_is = fig.add_subplot(gs[1, 1])

def style_slide_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#94A3B8")
    ax.spines["bottom"].set_color("#94A3B8")
    ax.tick_params(colors=tick_color)

# Subplot A: metriche PRDC sulla classe positiva.
prdc_labels = ["Precision", "Recall", "Density", "Coverage"]
ft_values = np.array([
    ft_row["precision"], ft_row["recall"], ft_row["density"], ft_row["coverage"]
], dtype=float)
fs_values = np.array([
    fs_row["precision"], fs_row["recall"], fs_row["density"], fs_row["coverage"]
], dtype=float)

x = np.arange(len(prdc_labels))
width = 0.38
ft_bars = ax_prdc.bar(x - width / 2, ft_values, width, label="Fine-tuned", color=ft_color)
fs_bars = ax_prdc.bar(x + width / 2, fs_values, width, label="From-scratch", color=fs_color)
ax_prdc.bar_label(ft_bars, fmt="%.2f", padding=3, fontsize=9, color=label_color)
ax_prdc.bar_label(fs_bars, fmt="%.2f", padding=3, fontsize=9, color=label_color)
ax_prdc.set_title("Metriche PRDC (classe positiva, n=1361)", fontsize=12, color=navy, pad=10)
ax_prdc.set_xticks(x)
ax_prdc.set_xticklabels(prdc_labels, fontsize=10, color=tick_color)
ax_prdc.tick_params(axis="y", labelsize=8, colors=tick_color)
ax_prdc.set_ylim(0, 1.20)
ax_prdc.grid(axis="y", color=grid_color, alpha=0.6, linewidth=0.8)
ax_prdc.set_axisbelow(True)
ax_prdc.legend(
    fontsize=10,
    frameon=False,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 1.18),
    handlelength=0.8,
    handletextpad=0.4,
    columnspacing=0.9,
)
style_slide_axis(ax_prdc)

# Subplot B: FID, con scala assoluta e barre orizzontali compatte.
fid_values = [float(fs_row["FID"]), float(ft_row["FID"])]
fid_bars = ax_fid.barh([1, 0], fid_values, color=bar_navy, height=0.36)
ax_fid.set_title("FID \u2014 pi\u00f9 basso \u00e8 meglio", fontsize=10, color=navy, pad=6)
ax_fid.set_yticks([0, 1])
ax_fid.set_yticklabels(["Fine-tuned", "From-scratch"], fontsize=9, color=tick_color)
ax_fid.bar_label(fid_bars, fmt="%.1f", padding=4, fontsize=9, color=label_color)
ax_fid.set_xlim(0, max(fid_values) * 1.20)
ax_fid.tick_params(axis="x", labelsize=7, colors=tick_color, rotation=45)
ax_fid.grid(False)
style_slide_axis(ax_fid)

# Subplot C: Inception Score, zoomato sul range utile per la slide.
is_values = [float(fs_row["IS_mean"]), float(ft_row["IS_mean"])]
is_bars = ax_is.barh([1, 0], is_values, color=bar_navy, height=0.36)
ax_is.set_title("Inception Score \u2014 pi\u00f9 alto \u00e8 meglio", fontsize=10, color=navy, pad=6)
ax_is.set_yticks([0, 1])
ax_is.set_yticklabels(["Fine-tuned", "From-scratch"], fontsize=9, color=tick_color)
ax_is.bar_label(is_bars, fmt="%.2f", padding=4, fontsize=9, color=label_color)
ax_is.set_xlim(2.20, max(is_values) * 1.015)
ax_is.tick_params(axis="x", labelsize=7, colors=tick_color)
ax_is.grid(False)
style_slide_axis(ax_is)

output_path = RESULTS_PLOTS_DIR / "final_test_metrics_slide_layout.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print("Salvato:", output_path)

## 7. Variazione filtro vs RAW per la slide D1

Questa figura compatta riassume il delta orientato al miglioramento tra campioni raw e filtrati.

In [ ]:
filter_metrics_path = RESULTS_FT_DIR / "metrics" / "validation_comparison_100_steps_raw_matched_vs_filtered.csv"
df_filter_metrics = pd.read_csv(filter_metrics_path)
df_filter_metrics = df_filter_metrics[df_filter_metrics["comparison_role"] == "filter_comparison"]

metric_order = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
metric_labels = ["FID (segno inv.)", "IS_mean", "Precision", "Recall", "Density", "Coverage"]
classes = ["negative", "positive"]
class_labels = {
    "negative": "Classe negativa",
    "positive": "Classe positiva",
}
class_colors = {
    "negative": "#6E86BF",
    "positive": "#263B70",
}

delta_by_class = {}
for class_name in classes:
    raw_row = df_filter_metrics[
        (df_filter_metrics["class"] == class_name)
        & (df_filter_metrics["stage"] == "raw_matched")
    ].iloc[0]
    filtered_row = df_filter_metrics[
        (df_filter_metrics["class"] == class_name)
        & (df_filter_metrics["stage"] == "filtered")
    ].iloc[0]

    deltas = []
    for metric in metric_order:
        raw_value = float(raw_row[metric])
        filtered_value = float(filtered_row[metric])
        if metric == "FID":
            delta = (raw_value - filtered_value) / raw_value * 100
        else:
            delta = (filtered_value - raw_value) / raw_value * 100
        deltas.append(delta)
    delta_by_class[class_name] = np.array(deltas, dtype=float)

fig, ax = plt.subplots(figsize=(8.4, 6.0), dpi=160)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

y = np.arange(len(metric_labels))
bar_height = 0.30
bars_negative = ax.barh(
    y - bar_height / 2,
    delta_by_class["negative"],
    height=bar_height,
    color=class_colors["negative"],
    label=class_labels["negative"],
)
bars_positive = ax.barh(
    y + bar_height / 2,
    delta_by_class["positive"],
    height=bar_height,
    color=class_colors["positive"],
    label=class_labels["positive"],
)

fig.suptitle(
    "Variazione % orientata al miglioramento \u2014 filtrate vs RAW",
    fontsize=12,
    color=navy,
    y=0.98,
)
ax.set_yticks(y)
ax.set_yticklabels(metric_labels, fontsize=11, color="#64748B")
ax.invert_yaxis()
ax.set_xlim(-30, 70)
ax.set_xticks(np.arange(-30, 71, 10))
ax.set_xticklabels([f"{tick:.0f}%" for tick in np.arange(-30, 71, 10)], fontsize=9, color="#64748B")
ax.axvline(0, color="#9CA3AF", linewidth=1.2)
ax.grid(axis="x", color=grid_color, alpha=0.75, linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.03),
    ncol=2,
    frameon=False,
    fontsize=9,
    handlelength=0.8,
    columnspacing=1.0,
)

for bars in [bars_negative, bars_positive]:
    for bar in bars:
        value = bar.get_width()
        if value >= 0:
            x_text = value + 1.0
            ha = "left"
        else:
            x_text = value - 1.0
            ha = "right"
        ax.text(
            x_text,
            bar.get_y() + bar.get_height() / 2,
            f"{value:+.1f}%",
            va="center",
            ha=ha,
            fontsize=8,
            color=label_color,
        )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#94A3B8")
ax.spines["bottom"].set_color("#94A3B8")
ax.tick_params(axis="y", length=0)
fig.subplots_adjust(top=0.86)

output_path = RESULTS_PLOTS_DIR / "filter_oriented_percent_delta_slide_layout.png"
fig.savefig(output_path, dpi=160, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print("Salvato:", output_path)